In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as transforms
from typing import Dict, List, Tuple, Optional
import logging
from pathlib import Path
from tqdm import tqdm
import numpy as np
import os
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import cv2
import seaborn as sns
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import train_test_split
import random
import wandb
%matplotlib inline

In [14]:
class LungCancerDataset(Dataset):
    
    def __init__(self, data_dir, transform=None):
        self.data_dir = data_dir
        self.transform = transform
        self.classes = ['Bengin cases', 'Malignant cases', 'Normal cases']
        self.class_to_idx = {cls: idx for idx, cls in enumerate(self.classes)}

        self.images = []
        self.labels = []

        # Load all images and labels
        for class_name in self.classes:
            class_dir = os.path.join(data_dir, class_name)
            if os.path.exists(class_dir):
                for img_file in os.listdir(class_dir):
                    if img_file.lower().endswith(('.png', '.jpg', '.jpeg')):
                        self.images.append(os.path.join(class_dir, img_file))
                        self.labels.append(self.class_to_idx[class_name])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, label

In [15]:
class CLAHETransform:
    """Apply CLAHE (Contrast Limited Adaptive Histogram Equalization)"""
    def __init__(self, clip_limit=2.0, tile_grid_size=(8, 8)):
        self.clip_limit = clip_limit
        self.tile_grid_size = tile_grid_size

    def __call__(self, image):
        if isinstance(image, Image.Image):
            image = np.array(image)

        # Convert to LAB and apply CLAHE to L channel
        if len(image.shape) == 3:  
            lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)
            clahe = cv2.createCLAHE(clipLimit=self.clip_limit, tileGridSize=self.tile_grid_size)
            lab[:,:,0] = clahe.apply(lab[:,:,0])
            image = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
        else:  
            clahe = cv2.createCLAHE(clipLimit=self.clip_limit, tileGridSize=self.tile_grid_size)
            image = clahe.apply(image)

        return Image.fromarray(image)

def get_data_transforms():
    train_transforms = transforms.Compose([
        CLAHETransform(clip_limit=2.0, tile_grid_size=(8, 8)),   # CLAHE
        transforms.Resize((224, 224)),                           # Resize
        transforms.RandomRotation(15),                           # Rotation
        transforms.RandomAffine(degrees=0,                       # Shift, zoom, shear
                                translate=(0.1, 0.1), 
                                scale=(0.9, 1.1), 
                                shear=10, 
                                fill=0),                        # fill pixels with 0 (black)
        transforms.RandomHorizontalFlip(p=0.5),                  # Flip
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                             std=[0.229, 0.224, 0.225])
    ])

    test_transforms = transforms.Compose([
        CLAHETransform(clip_limit=2.0, tile_grid_size=(8, 8)),   # CLAHE
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                             std=[0.229, 0.224, 0.225])
    ])

    return train_transforms, test_transforms




In [16]:
config = {
    'data_dir': '/kaggle/input/the-iqothnccd-lung-cancer-dataset/The IQ-OTHNCCD lung cancer dataset',
    'batch_size': 32,
    'learning_rate': 0.001,
    'epochs': 50,
    'num_heads': 4,
    'key_dim': 16,
    'val_split': 0.1,
    'test_split': 0.1,
    'seed': 42,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu'
}

In [17]:
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

train_transforms, test_transforms = get_data_transforms()
full_dataset = LungCancerDataset(config['data_dir'], transform=train_transforms)

dataset_size = len(full_dataset)
val_size = int(config['val_split'] * dataset_size)
test_size = int(config['test_split'] * dataset_size)
train_size = dataset_size - val_size - test_size

train_dataset, val_dataset, test_dataset = random_split(
    full_dataset, [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(config['seed'])
)

# Update val/test transforms
val_dataset.dataset.transform = test_transforms
test_dataset.dataset.transform = test_transforms

train_loader = DataLoader(train_dataset, batch_size=config['batch_size'], shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=config['batch_size'], shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=config['batch_size'], shuffle=False)

print(f"Dataset split: Train={len(train_dataset)}, Val={len(val_dataset)}, Test={len(test_dataset)}")

Dataset split: Train=879, Val=109, Test=109


In [18]:
class CNN1(nn.Module):
    def __init__(self):
        super().__init__()
        self.block1 = nn.Sequential(nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2))
        self.block2 = nn.Sequential(nn.Conv2d(32, 64, 3, padding=1), nn.ReLU())
        self.block3 = nn.Sequential(nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2))
        self.block4 = nn.Sequential(nn.Conv2d(64,128,3,padding=1), nn.ReLU())
        self.block5 = nn.Sequential(nn.Conv2d(128,128,3,padding=1), nn.ReLU(), nn.MaxPool2d(2))
        self.block6 = nn.Sequential(nn.Conv2d(128,256,3,padding=1), nn.ReLU())
    def forward(self,x):
        x = self.block1(x); x = self.block2(x); x = self.block3(x)
        x = self.block4(x); x = self.block5(x); x = self.block6(x)
        return x

class CNN2(nn.Module):
    def __init__(self):
        super().__init__()
        self.block1 = nn.Sequential(nn.Conv2d(3,16,3,padding=1), nn.ReLU(), nn.MaxPool2d(2))
        self.block2 = nn.Sequential(nn.Conv2d(16,32,3,padding=1), nn.ReLU(), nn.MaxPool2d(2))
    def forward(self,x):
        x = self.block1(x); x = self.block2(x)
        return x

class MainModel(nn.Module):
    def __init__(self, num_classes=3, num_heads=4, key_dim=16):
        super().__init__()
        self.cnn1 = CNN1()
        self.cnn2 = CNN2()
        self.to_embed = nn.Conv1d(288, 64, kernel_size=1)
        self.mha1 = nn.MultiheadAttention(embed_dim=64, num_heads=num_heads, batch_first=False)
        self.mha2 = nn.MultiheadAttention(embed_dim=64, num_heads=num_heads, batch_first=False)
        self.gap1d = nn.AdaptiveAvgPool1d(1)
        self.bn1 = nn.BatchNorm1d(64)
        self.fc1 = nn.Linear(64, 256)
        self.gelu = nn.GELU()
        self.bn2 = nn.BatchNorm1d(256)
        self.drop = nn.Dropout(0.2)
        self.fc_out = nn.Linear(256, num_classes)

    def forward(self, x):
        f1 = self.cnn1(x)
        f2 = self.cnn2(x)
        if f1.shape[-2:] != f2.shape[-2:]:
            f2 = F.interpolate(f2, size=f1.shape[-2:], mode='bilinear', align_corners=False)
        feats = torch.cat([f1, f2], dim=1)
        B,C,H,W = feats.shape
        seq = feats.view(B, C, H*W)
        seq = self.to_embed(seq)                  # (B,64,L)
        seq = seq.permute(2,0,1)                  # (L,B,64)
        seq,_ = self.mha1(seq,seq,seq)
        seq,_ = self.mha2(seq,seq,seq)
        seq = seq.permute(1,2,0)                  # (B,64,L)
        pooled = self.gap1d(seq).squeeze(-1)
        z = self.bn1(pooled)
        z = self.fc1(z); z = self.gelu(z); z = self.bn2(z); z = self.drop(z)
        return self.fc_out(z)


In [19]:
def train_model(model, train_loader, val_loader, num_epochs, learning_rate, device):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    train_losses, val_losses, train_accs, val_accs = [], [], [], []
    model.to(device)

    for epoch in range(num_epochs):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * images.size(0)
            _, preds = outputs.max(1)
            correct += preds.eq(labels).sum().item()
            total += labels.size(0)
        train_loss = running_loss / total
        train_acc = correct / total

        # Validation
        model.eval()
        val_running_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_running_loss += loss.item() * images.size(0)
                _, preds = outputs.max(1)
                val_correct += preds.eq(labels).sum().item()
                val_total += labels.size(0)
        val_loss = val_running_loss / val_total
        val_acc = val_correct / val_total

        train_losses.append(train_loss)
        val_losses.append(val_loss)
        train_accs.append(train_acc)
        val_accs.append(val_acc)

        print(f"Epoch {epoch+1}/{num_epochs}: "
              f"Train Loss={train_loss:.4f}, Train Acc={train_acc:.4f}, "
              f"Val Loss={val_loss:.4f}, Val Acc={val_acc:.4f}")

        wandb.log({
            "epoch": epoch+1,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc
        })

    return train_losses, val_losses, train_accs, val_accs

def evaluate_model(model, dataloader, device):
    model.eval()
    preds_all, labels_all = [], []
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = outputs.max(1)
            preds_all.extend(preds.cpu().numpy())
            labels_all.extend(labels.cpu().numpy())
    acc = accuracy_score(labels_all, preds_all)
    prec = precision_score(labels_all, preds_all, average='weighted')
    rec = recall_score(labels_all, preds_all, average='weighted')
    f1 = f1_score(labels_all, preds_all, average='weighted')
    try:
        auc = roc_auc_score(labels_all, F.one_hot(torch.tensor(preds_all), num_classes=3), multi_class='ovr')
    except:
        auc = None
    print(classification_report(labels_all, preds_all, target_names=['Bengin','Malignant','Normal']))
    return acc, prec, rec, f1, auc

def plot_training_history(train_losses, val_losses, train_accs, val_accs):
    plt.figure(figsize=(12,5))
    plt.subplot(1,2,1)
    plt.plot(train_losses, label="Train Loss")
    plt.plot(val_losses, label="Val Loss")
    plt.legend(); plt.title("Loss")
    plt.subplot(1,2,2)
    plt.plot(train_accs, label="Train Acc")
    plt.plot(val_accs, label="Val Acc")
    plt.legend(); plt.title("Accuracy")
    plt.show()

In [20]:
wandb.login(key='REDACTED_WANDB_API_KEY')  # Replace with your actual key

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: tahmidulkashfi15 (tahmidulkashfi15-international-islamic-university-chittagong) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
wandb.init(
    project="base_ccal",
    name="base_model_training",
    config=config
)

# Create Model
model = MainModel(num_classes=3, num_heads=config['num_heads'], key_dim=config['key_dim'])
print(f"Model Parameters: {sum(p.numel() for p in model.parameters()):,}")

wandb.watch(model, log="all", log_freq=100)

# Train
train_losses, val_losses, train_accs, val_accs = train_model(
    model, train_loader, val_loader,
    num_epochs=config['epochs'],
    learning_rate=config['learning_rate'],
    device=config['device']
)

plot_training_history(train_losses, val_losses, train_accs, val_accs)

# Evaluate
accuracy, precision, recall, f1, auc = evaluate_model(model, test_loader, device=config['device'])
wandb.log({"test_acc": accuracy, "test_precision": precision, "test_recall": recall, "test_f1": f1, "test_auc": auc})

# Save
torch.save(model.state_dict(), 'base_ccal.pth')
artifact = wandb.Artifact('base_ccal', type='model')
artifact.add_file('base_ccal.pth')
wandb.log_artifact(artifact)

wandb.finish()

Model Parameters: 647,843
Epoch 1/50: Train Loss=0.6294, Train Acc=0.7418, Val Loss=0.9875, Val Acc=0.4404
Epoch 2/50: Train Loss=0.4843, Train Acc=0.8146, Val Loss=1.2234, Val Acc=0.4404
Epoch 3/50: Train Loss=0.4389, Train Acc=0.8305, Val Loss=2.3874, Val Acc=0.5505
Epoch 4/50: Train Loss=0.4578, Train Acc=0.8191, Val Loss=10.4504, Val Acc=0.4128
Epoch 5/50: Train Loss=0.3975, Train Acc=0.8316, Val Loss=4.6156, Val Acc=0.4771
Epoch 6/50: Train Loss=0.3458, Train Acc=0.8555, Val Loss=0.4630, Val Acc=0.8349
Epoch 7/50: Train Loss=0.3772, Train Acc=0.8441, Val Loss=2.6799, Val Acc=0.5688
Epoch 8/50: Train Loss=0.3828, Train Acc=0.8385, Val Loss=1.3444, Val Acc=0.6055
Epoch 9/50: Train Loss=0.3934, Train Acc=0.8339, Val Loss=10.0565, Val Acc=0.4679
Epoch 10/50: Train Loss=0.3418, Train Acc=0.8498, Val Loss=6.8555, Val Acc=0.1743
Epoch 11/50: Train Loss=0.3469, Train Acc=0.8567, Val Loss=2.0199, Val Acc=0.5872
Epoch 12/50: Train Loss=0.3184, Train Acc=0.8612, Val Loss=3.1567, Val Acc=0.46